<a href="https://colab.research.google.com/github/AlixPriaRamadhani/03-02-2026/blob/main/csvKantin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
harga = np.array([5000, 7000, 3000, 12000, 4500])
print('Rata-rata harga:', harga.mean())
print('Harga tertinggi:', harga.max())
print('Harga setelah diskon 10%:', harga * 0.9)

Rata-rata harga: 6300.0
Harga tertinggi: 12000
Harga setelah diskon 10%: [ 4500.  6300.  2700. 10800.  4050.]


In [3]:
harga_list = [5000, 7000, 3000]
print(harga_list * 0.9)

TypeError: can't multiply sequence by non-int of type 'float'

NumPy array mendukung operasi vektor/elementwise secara native, sedangkan list Python tidak — operator * pada list berarti replikasi, bukan perkalian matematis, dan itu hanya bekerja dengan integer, bukan float.

In [4]:
import pandas as pd
data_kantin = {
'menu': ['Nasi Goreng', 'Es Teh', 'Mie Ayam', 'Es Teh', None],
'harga': [12000, 4000, 10000, 4000, 8000],
'terjual': [23, 40, None, 35, 18]
}
df = pd.DataFrame(data_kantin)
print(df)

          menu  harga  terjual
0  Nasi Goreng  12000     23.0
1       Es Teh   4000     40.0
2     Mie Ayam  10000      NaN
3       Es Teh   4000     35.0
4         None   8000     18.0


data kosong harus ditangani dulu (misalnya dengan dropna() untuk menghapus baris, atau fillna() untuk mengisi nilai default) sebelum dianalisis, supaya hasil analisis akurat dan tidak menyesatkan.

In [7]:
print(df.head()) # 5 baris pertama
print(df.info()) # tipe data & jumlah non-null tiap kolom
print(df.describe()) # statistik ringkas kolom numerik
print(df.shape) # jumlah (baris, kolom)

          menu  harga  terjual
0  Nasi Goreng  12000     23.0
1       Es Teh   4000     40.0
2     Mie Ayam  10000      NaN
3       Es Teh   4000     35.0
4         None   8000     18.0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   menu     4 non-null      object 
 1   harga    5 non-null      int64  
 2   terjual  4 non-null      float64
dtypes: float64(1), int64(1), object(1)
memory usage: 252.0+ bytes
None
              harga    terjual
count      5.000000   4.000000
mean    7600.000000  29.000000
std     3577.708764  10.230673
min     4000.000000  18.000000
25%     4000.000000  21.750000
50%     8000.000000  29.000000
75%    10000.000000  36.250000
max    12000.000000  40.000000
(5, 3)


menu (4 non-null dari 5 baris)
terjual (4 non-null dari 5 baris)

Kedua kolom tersebut masing-masing memiliki 1 data kosong (missing value / NaN). Ini menandakan bahwa dataset belum bersih sepenuhnya, sehingga perlu ditangani (misalnya dengan dropna() atau fillna()) sebelum dilakukan analisis lebih lanjut, agar hasil perhitungan (seperti total pendapatan atau rata-rata) tidak salah atau bias.

In [8]:
print(df.isnull().sum()) # jumlah data kosong tiap kolom
df['terjual'] = df['terjual'].fillna(0) # isi kekosongan dengan 0
df = df.dropna(subset=['menu']) # hapus baris jika kolom menu kosong

menu       1
harga      0
terjual    1
dtype: int64


karena jenis data dan dampaknya berbeda, sehingga penanganannya juga harus berbeda:

Jika data kosong bisa diberi nilai default yang logis dan tidak mengubah makna data → gunakan fillna().
Jika data kosong berada di kolom yang krusial sebagai identitas/kategori dan tidak ada nilai pengganti yang masuk akal → gunakan dropna(), karena mempertahankan baris tersebut justru bisa menyesatkan analisis.

In [9]:
print(df.duplicated().sum()) # jumlah baris duplikat
df = df.drop_duplicates()
df['harga'] = df['harga'].astype(int) # memastikan tipe data harga adalah integer
print(df.dtypes)

0
menu        object
harga        int64
terjual    float64
dtype: object


Memastikan dtypes benar adalah langkah validasi penting sebelum analisis, supaya semua operasi (matematis, filter, agregasi) berjalan sesuai yang diharapkan dan hasilnya bisa dipercaya.

In [10]:
laris = df[df['terjual'] > 20] # filtering
urut = df.sort_values(by='terjual', ascending=False) # sorting
df['total_pendapatan'] = df['harga'] * df['terjual'] # kolom turunan
ringkasan = df.groupby('menu')['total_pendapatan'].sum() # agregasi
print(ringkasan)

menu
Es Teh         300000.0
Mie Ayam            0.0
Nasi Goreng    276000.0
Name: total_pendapatan, dtype: float64


Menu dengan total_pendapatan tertinggi adalah Es Teh, meskipun harganya paling murah dibanding menu lain. Ini terjadi karena total pendapatan ditentukan oleh harga × jumlah terjual, bukan harga saja — dan Es Teh punya volume penjualan yang jauh lebih tinggi dibanding menu lainnya.